# BotGuard: Advanced Hybrid Bot Detector Training Notebook
### *Zero-Day Bot Detection using Bi-Directional GRU + GraphSAGE with Parallel Dual-T4 Hyperparameter Tuning (Optuna)*


---

## Foundations
1. **CALEB (arXiv:2205.15707)**: A Conditional Adversarial Learning Framework to Enhance Bot Detection. We combat Zero-Day evasion tactics (temporal smoothing, semantic camouflaging) by structuring our training for active learning robustness.
2. **Cresci et al. (2017)**: Highlighting that bots show highly correlated and asymmetric structural features (e.g., highly skewed follower-to-following ratios, low-entropy action logs) compared to human power-law distributions.

## Neural Network Architecture
Our model fuses spatial and temporal social representations to make high-fidelity classifications:
- **Temporal Encoder (Bi-directional GRU)**: Processes the chronologically ordered sequence of a user's latest actions/tweets. Learns temporal patterns and text complexity features ($D_{\text{in}} = 2$).
- **Spatial Encoder (GraphSAGE)**: Aggregates graph topological features from the user's followers/following neighborhood. Extracts asymmetrical social structures ($D_{\text{in}} = 3$).
- **Hybrid Classifier Head**: Concat-fuses the final representations and applies fully connected layers with heavy regularization (Dropout, Weight Decay) to output a bot probability ($P \in [0, 1]$).

```
  Temporal Seq (B, L, 2) --> [Bi-GRU Encoder] --------+ 
                                                        |--> Fusion --> [Classifier Head] --> Bot Prob
  Graph (X, edge_index)  --> [GraphSAGE Encoder] [Target]-+
```

In [ ]:
# Install required dependencies for Graph Neural Networks and Hyperparameter Optimization
!pip install -q torch-geometric optuna matplotlib seaborn scikit-learn

In [ ]:
import os
import json
import random
import logging
import threading
import contextlib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
set_seed(42)

## Model Definition
We define the precise architectures for `TemporalEncoder`, `SpatialEncoder`, and `HybridBotDetector` to guarantee absolute compatibility with the production code. We implement a flexible dimension check in `HybridBotDetector` to seamlessly handle both batch training tensors and single-integer API inference requests.

In [ ]:
class TemporalEncoder(nn.Module):
    """
    Bi-directional GRU to process the sequence of recent user actions.
    Captures temporal entropy and action frequency.
    """
    def __init__(self, input_dim: int, hidden_dim: int):
        super(TemporalEncoder, self).__init__()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        
        output, hidden = self.gru(x)
        
        forward_hidden = hidden[-2, :, :]
        backward_hidden = hidden[-1, :, :]
        return torch.cat((forward_hidden, backward_hidden), dim=1)


class SpatialEncoder(nn.Module):
    """
    GraphSAGE to process the user's neighborhood topology.
    Captures follower/following asymmetry and isolation.
    """
    def __init__(self, in_channels: int, hidden_channels: int, out_channels: int, dropout: float = 0.3):
        super(SpatialEncoder, self).__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        # x shape: (num_nodes, in_channels)
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x


class HybridBotDetector(nn.Module):
    """
    Combines Spatial and Temporal features to classify users as Bot or Human.
    """
    def __init__(self, temporal_in: int, spatial_in: int, hidden_dim: int = 64, dropout: float = 0.3):
        super(HybridBotDetector, self).__init__()
        
        self.temporal_encoder = TemporalEncoder(input_dim=temporal_in, hidden_dim=hidden_dim)
        self.spatial_encoder = SpatialEncoder(
            in_channels=spatial_in, 
            hidden_channels=hidden_dim, 
            out_channels=hidden_dim * 2,
            dropout=dropout
        )
        
        # Classifier head: (Bi-GRU output = 2*hidden) + (GraphSAGE output = 2*hidden) -> 4*hidden
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, temporal_seq: torch.Tensor, node_features: torch.Tensor, edge_index: torch.Tensor, target_node_idx) -> torch.Tensor:
        temporal_emb = self.temporal_encoder(temporal_seq) 
        
        spatial_emb_all = self.spatial_encoder(node_features, edge_index)
        

        if isinstance(target_node_idx, (int, np.integer)):
            spatial_emb_target = spatial_emb_all[target_node_idx].unsqueeze(0)
        elif isinstance(target_node_idx, torch.Tensor) and target_node_idx.dim() == 0:
            spatial_emb_target = spatial_emb_all[target_node_idx].unsqueeze(0)
        else:
            spatial_emb_target = spatial_emb_all[target_node_idx]
        
        fused_features = torch.cat((temporal_emb, spatial_emb_target), dim=1)
        probability = self.classifier(fused_features)
        
        return probability

## Data Ingestion & Preprocessing
Because standard counts (followers, following) show extreme scale variances, we apply a mathematical log-normalization ($x = \log(1 + count)$) to stabilize the gradient flow.

If labels are missing or set to `None`, we employ a **Cresci-inspired Scientific Heuristic** to construct clean observed labels $\tilde{y}$ for booting up our ML loop.

In [ ]:
def preprocess_twibot_data(json_path):
    logger.info(f"Loading TwiBot-20 dataset from: {json_path}")
    with open(json_path, 'r') as f:
        data = json.load(f)
    logger.info(f"Loaded {len(data)} user records successfully.")
    
    
    user_id_to_idx = {user['ID'].strip(): idx for idx, user in enumerate(data)}
    
    node_features = []
    temporal_features = []
    labels = []
    
    for idx, user in enumerate(data):
        profile = user.get('profile', {})
        followers = float(profile.get('followers_count', 0) or 0)
        friends = float(profile.get('friends_count', 0) or 0)
        
        # Apply Log-normalization
        log_followers = np.log1p(followers)
        log_friends = np.log1p(friends)
        ratio = followers / (friends + 1.0)
        log_ratio = np.log1p(ratio)
        
        node_features.append([log_followers, log_friends, log_ratio])
        
        
        tweets = user.get('tweet', [])
        if tweets is None:
            tweets = []
        
        seq = []
        for i in range(10):
            if i < len(tweets):
                tweet = tweets[i]
                if not isinstance(tweet, str):
                    tweet = str(tweet)
                # Feature 1: Normalized length
                len_feat = min(len(tweet) / 280.0, 1.0)
                # Feature 2: Is Retweet or contains Link/Hashtag
                is_complex = 1.0 if (tweet.startswith("RT @") or "http" in tweet or "#" in tweet) else 0.0
                seq.append([len_feat, is_complex])
            else:
                seq.append([0.0, 0.0]) # Zero padding for users with fewer than 10 tweets
        temporal_features.append(seq)
        
        
        label = user.get('label')
        if label is None:
           # Detect automated behaviors based on extreme asymmetric networking
            screen_name = str(profile.get('screen_name', '')).lower()
            digit_count = len([c for c in screen_name if c.isdigit()])
            
            if friends > 5 * (followers + 1) and friends > 100:
                labels.append(1) # Bot
            elif digit_count >= 4 and followers < 50:
                labels.append(1) # Bot
            else:
                labels.append(0) # Human
        else:
            if isinstance(label, str):
                labels.append(1 if label.lower() == 'bot' else 0)
            else:
                labels.append(int(label))
                
    # Construct directed Edge Index representing followers/following connections
    edge_starts = []
    edge_ends = []
    for idx, user in enumerate(data):
        neighbor = user.get('neighbor')
        if isinstance(neighbor, dict):
            for fol_id in neighbor.get('follower', []):
                fol_id_str = str(fol_id).strip()
                if fol_id_str in user_id_to_idx:
                    edge_starts.append(user_id_to_idx[fol_id_str])
                    edge_ends.append(idx)
            for folg_id in neighbor.get('following', []):
                folg_id_str = str(folg_id).strip()
                if folg_id_str in user_id_to_idx:
                    edge_starts.append(idx)
                    edge_ends.append(user_id_to_idx[folg_id_str])
        
        # Add self-loops to guarantee graph connectivity and SAGEConv message aggregation stability
        edge_starts.append(idx)
        edge_ends.append(idx)
        
    dataset = {
        'node_features': torch.tensor(node_features, dtype=torch.float32),
        'temporal_features': torch.tensor(temporal_features, dtype=torch.float32),
        'edge_index': torch.tensor([edge_starts, edge_ends], dtype=torch.long),
        'labels': torch.tensor(labels, dtype=torch.float32)
    }
    
    logger.info(f"Processed {dataset['node_features'].shape[0]} nodes and {dataset['edge_index'].shape[1]} edges.")
    logger.info(f"Bootstrap labels breakdown: {np.bincount(labels)}")
    return dataset

In [ ]:
class GPUPool:
    """
    Thread-safe GPU resources manager designed for parallelizing Optuna hyperparameter searches
    across multiple local T4 GPUs (cuda:0, cuda:1).
    """
    def __init__(self):
        num_gpus = torch.cuda.device_count()
        self.gpu_ids = list(range(num_gpus)) if num_gpus > 0 else []
        self.lock = threading.Lock()
        self.allocated = {gpu_id: False for gpu_id in self.gpu_ids}
        logger.info(f"GPUPool initialized with physical devices: {self.gpu_ids}")

    @contextlib.contextmanager
    def acquire(self):
        if not self.gpu_ids:
            yield "cpu"
            return
            
        gpu_id = None
        while gpu_id is None:
            with self.lock:
                for gid, is_busy in self.allocated.items():
                    if not is_busy:
                        self.allocated[gid] = True
                        gpu_id = gid
                        break
            if gpu_id is None:
                import time
                time.sleep(0.1)
        try:
            device = f"cuda:{gpu_id}"
            yield device
        finally:
            with self.lock:
                self.allocated[gpu_id] = False

gpu_pool = GPUPool()

## Training Loop with Early Stopping & L2 Regularization
To completely prevent **overfitting**, our loop implements three layers of control:
1. **Early Stopping**: Halts execution if validation F1 score does not improve for 15 consecutive epochs.
2. **L2 Weight Regularization**: Configured in `AdamW` to decay weights during gradient steps.
3. **Learning Rate Scheduler**: A `ReduceLROnPlateau` decays learning rate when loss stalls.

In [ ]:
def train_evaluate_trial(trial_params, dataset, indices, device):
    """
    Trains a HybridBotDetector configuration on the designated device, returning
    the validation F1-score for hyperparameter decisions.
    """

    train_idx, val_idx = indices['train'], indices['val']
    

    hidden_dim = trial_params['hidden_dim']
    lr = trial_params['lr']
    weight_decay = trial_params['weight_decay']
    dropout = trial_params['dropout']
    

    model = HybridBotDetector(
        temporal_in=2,
        spatial_in=3,
        hidden_dim=hidden_dim,
        dropout=dropout
    ).to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    criterion = nn.BCELoss()
    

    node_features = dataset['node_features'].to(device)
    edge_index = dataset['edge_index'].to(device)
    labels = dataset['labels'].to(device)
    temporal_features = dataset['temporal_features'].to(device)
    

    train_t = torch.tensor(train_idx, dtype=torch.long, device=device)
    val_t = torch.tensor(val_idx, dtype=torch.long, device=device)
    
    best_val_f1 = 0.0
    epochs_no_improve = 0
    patience = 15
    

    for epoch in range(80):
        model.train()
        optimizer.zero_grad()
        

        batch_temp = temporal_features[train_t] # (B, 10, 2)
        pred = model(batch_temp, node_features, edge_index, train_t).squeeze()
        
        loss = criterion(pred, labels[train_t])
        loss.backward()
        optimizer.step()
        
        # Evaluation
        model.eval()
        with torch.no_grad():
            val_temp = temporal_features[val_t]
            val_pred = model(val_temp, node_features, edge_index, val_t).squeeze()
            val_pred_binary = (val_pred >= 0.5).float()
            

            val_pred_np = val_pred_binary.cpu().numpy()
            val_labels_np = labels[val_t].cpu().numpy()
            

            val_f1 = f1_score(val_labels_np, val_pred_np, zero_division=0)
            

        scheduler.step(val_f1)
        

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break
                
    return best_val_f1

## Optuna Tuning
We construct our Optuna Study, targeting `n_jobs=2` to trigger concurrent multi-threaded execution. Each trial safely acquires a dedicated GPU from our GPUPool to run parallel neural network updates!

In [ ]:

json_file_path = "twibot20.json" # Set matching local or Kaggle data location
if not os.path.exists(json_file_path):

    json_file_path = "/kaggle/input/twibot20/twibot20.json" if os.path.exists("/kaggle/input/twibot20/twibot20.json") else "twibot20.json"

if not os.path.exists(json_file_path):
    raise FileNotFoundError("Dataset file 'twibot20.json' was not found! Please check paths.")

dataset = preprocess_twibot_data(json_file_path)
num_nodes = dataset['node_features'].shape[0]


all_indices = np.arange(num_nodes)
train_idx, temp_idx = train_test_split(all_indices, test_size=0.30, random_state=42, stratify=dataset['labels'].numpy())
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=42, stratify=dataset['labels'].numpy()[temp_idx])

split_indices = {
    'train': train_idx,
    'val': val_idx,
    'test': test_idx
}

def optuna_objective(trial):

    trial_params = {
        'hidden_dim': trial.suggest_categorical('hidden_dim', [32, 64, 128]),
        'lr': trial.suggest_float('lr', 1e-4, 1e-2, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True),
        'dropout': trial.suggest_float('dropout', 0.1, 0.6)
    }
    

    with gpu_pool.acquire() as device:
        score = train_evaluate_trial(trial_params, dataset, split_indices, device)
        
    return score

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)


study = optuna.create_study(direction="maximize")
logger.info("Launching Parallel Hyperparameter Optimization on GPUPool...")


study.optimize(optuna_objective, n_trials=40, n_jobs=min(2, max(1, torch.cuda.device_count())))

logger.info("Optimization complete!")
logger.info(f"Best trial score (F1): {study.best_value:.4f}")
logger.info(f"Best hyperparameters: {study.best_params}")

## Final Model Retraining & Test Validation
Now, we build the absolute best model config on the primary T4 GPU, train it, evaluate performance on our untouched **Test Set** using multi-dimensional scientific metrics (Accuracy, F1, Precision, Recall, and ROC-AUC), and serialize the final trained weights as **`twibot_baseline.pt`**.

In [ ]:
best_params = study.best_params
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

logger.info(f"Retraining best configuration on {device}...")
model = HybridBotDetector(
    temporal_in=2,
    spatial_in=3,
    hidden_dim=best_params['hidden_dim'],
    dropout=best_params['dropout']
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=best_params['lr'], 
    weight_decay=best_params['weight_decay']
)
criterion = nn.BCELoss()


node_features = dataset['node_features'].to(device)
edge_index = dataset['edge_index'].to(device)
labels = dataset['labels'].to(device)
temporal_features = dataset['temporal_features'].to(device)

train_t = torch.tensor(split_indices['train'], dtype=torch.long, device=device)
val_t = torch.tensor(split_indices['val'], dtype=torch.long, device=device)
test_t = torch.tensor(split_indices['test'], dtype=torch.long, device=device)


train_losses = []
val_losses = []

for epoch in range(300):
    model.train()
    optimizer.zero_grad()
    
    pred = model(temporal_features[train_t], node_features, edge_index, train_t).squeeze()
    loss = criterion(pred, labels[train_t])
    loss.backward()
    optimizer.step()
    
    # Tracks val loss
    model.eval()
    with torch.no_grad():
        val_pred = model(temporal_features[val_t], node_features, edge_index, val_t).squeeze()
        val_loss = criterion(val_pred, labels[val_t])
        
    train_losses.append(loss.item())
    val_losses.append(val_loss.item())
    
# Evaluate on Untouched Test Set
model.eval()
with torch.no_grad():
    test_pred = model(temporal_features[test_t], node_features, edge_index, test_t).squeeze()
    test_pred_binary = (test_pred >= 0.5).float()
    
    test_pred_np = test_pred_binary.cpu().numpy()
    test_scores_np = test_pred.cpu().numpy()
    test_labels_np = labels[test_t].cpu().numpy()
    
    accuracy = accuracy_score(test_labels_np, test_pred_np)
    precision = precision_score(test_labels_np, test_pred_np, zero_division=0)
    recall = recall_score(test_labels_np, test_pred_np, zero_division=0)
    f1 = f1_score(test_labels_np, test_pred_np, zero_division=0)
    
    try:
        auc_score = roc_auc_score(test_labels_np, test_scores_np)
    except Exception:
        auc_score = 0.5 

print(f"\n{'='*40}\nFINAL METRICS ON UNTOUCHED TEST SET:\n{'='*40}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {auc_score:.4f}")
print(f"{'='*40}")

# Plot loss curves
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', color='#4F46E5', linewidth=2.5)
plt.plot(val_losses, label='Validation Loss', color='#10B981', linewidth=2.5)
plt.title('Training Progression (Loss Curves)', fontsize=14, fontweight='bold')
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('loss_curves.png')
plt.show()

# Export model weights 
export_path = "twibot_baseline.pt"
torch.save(model.state_dict(), export_path)
print(f"Successfully exported calibrated state weights to: {export_path}")